# 🎯 [DAT10_M06] 오프라인/온라인 전환 상태 및 재고 변동 내역 실시간 대시보드 UI 구현
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT10/dat10_m06_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT10_M06 |
| **미션 주제** | 오프라인/온라인 전환 상태 및 재고 변동 내역 실시간 대시보드 UI 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
전시에는 전술 통신망이 단절된 채 예하 보급소가 입고와 불출을 각자 기록하고, 통신 복구 후 기록을 병합하는 과정에서 중복·누락·충돌·위변조가 빈번히 발생하며, 이를 적시에 통제하지 못하면 재고 왜곡과 방산 공급망 위기 식별이 지연됩니다. 본 프로젝트는 방산 공급망 회복력 연구(미국 방위산업기반 정책연구)의 문제의식을 교육용 가상 자료로 재구성해, 가상 보급소 5곳이 통신 두절 환경에서 오프라인으로 재고를 기록하고 동기화하는 SaaS를 개발하고, 동기화된 데이터에서 공급망 재난 징후(업체 공급 감소, 재고 소진 임박)를 식별하는 End-to-End 군수 소프트웨어 엔지니어링 과제입니다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 06. 오프라인/온라인 전환 상태 및 재고 변동 내역 실시간 대시보드 UI 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

오프라인/온라인 상태 및 재고 변동 내역 실시간 대시보드 UI를 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **보급소별 오프라인/온라인 상태와 품목별 재고 변동 내역, 공급 경보를 보여 주는 실시간 대시보드 화면 모델을 구현하고 갱신 1초 이내·조작 응답 100ms 이내를 확인한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **7일치 트랜잭션 2,100건을 1시간 단위 168회 데이터 스트림으로 흘려 보급소 상태판(온라인·오프라인·미동기 건수·지연 시간), 품목별 재고 게이지(소진 예상 일수와 납기 비교), 변동 내역, 경보 목록 4개 패널의 화면 모델을 갱신 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ui/dat10_dashboard.py (또는 웹 UI 컴포넌트)` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 전술 작전 상황판으로서의 직관성 및 인지 부하 감소도

#### 미션 PASS 조건

- 정량 기준: 화면 갱신 주기 1초 이내, UI 응답성 100ms 이내


- 진행 기준: 주요 지표 시각화 컴포넌트 4종 이상 렌더링 및 실시간 데이터 연동


- 검증 방법: 사용자 인터랙션 테스트 및 모의 데이터 스트림 렌더링 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""5개 보급소의 오프라인/온라인 전환 상태와 품목별 재고 변동 내역을 시간 단위 모의 데이터 스트림으로 갱신하는 실시간 대시보드 화면 모델(상태판·재고 게이지·변동 내역·경보 패널 4종)과 야간 모드·긴급 경보 강조·조작 응답 시간을 구현해 갱신 주기 1초 이내·UI 응답 100ms 이내를 측정한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 야전 보급소·업체·품목 자료 생성기 (실제 부대·업체·품목·재고 정보 없음)
# 가상 보급소 5곳이 품목 12종의 입고·불출·조정 트랜잭션을 오프라인으로 기록하고 나중에 동기화한다.
# 공급망 재난 징후(업체 납기 지연, 품목 재고 급감)는 동기화된 자료를 마이닝해 식별한다.
# ---------------------------------------------------------------------------
NODES = ["보급소A", "보급소B", "보급소C", "보급소D", "보급소E"]
SUPPLIERS = ["가상업체1", "가상업체2", "가상업체3", "가상업체4", "가상업체5", "가상업체6"]
CATS = ["일반보급", "정비부품", "의무소모"]
ITEMS = [{"item": f"I{i + 1:02d}", "cat": CATS[i % 3], "supplier": SUPPLIERS[i % 6], "lead_days": 3 + (i * 5) % 12, "use": 4 + (i * 7) % 11, "min_level": 30 + 10 * (i % 5)} for i in range(12)]
ITEM_IDS = [x["item"] for x in ITEMS]
CTXS = ["평시", "야간", "통신불량"]


def make_txns(seed=7, days=7, per_node_day=60, nodes=None):
    """노드별 오프라인 트랜잭션 목록. id는 노드-일련번호(멱등 키), t는 시각(시간 단위), ctx는 입력 상황."""
    rng = np.random.default_rng(seed)
    out = []
    for n in (nodes or NODES):
        seq = 0
        for k in range(days * per_node_day):
            seq += 1
            it = ITEMS[int(rng.integers(len(ITEMS)))]
            t = float(rng.uniform(0, 24 * days))
            ctx = "야간" if (t % 24 >= 20 or t % 24 < 5) else "평시"
            if rng.random() < 0.12:
                ctx = "통신불량"
            r = rng.random()
            if r < 0.22:
                op, q = "in", int(it["use"] * rng.uniform(4, 9))
            elif r < 0.95:
                op, q = "out", int(max(1, rng.normal(it["use"] * 1.5, 3)))
            else:
                op, q = "adjust", int(rng.integers(-5, 6))
            out.append({"id": f"{n}-{seq:05d}", "node": n, "item": it["item"], "op": op, "qty": q, "t": round(t, 2), "ctx": ctx, "supplier": it["supplier"]})
    return out


def stock_of(txns, base=500):
    """품목별 재고 = 기초 재고 + 입고 - 불출 + 조정 (id 중복은 한 번만 반영)."""
    # [TODO : 학생 실습 6] 오프라인/온라인 전환 상태 및 재고 변동 내역 실시간 대시보드 UI 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
THEMES = {"주간": {"bg": (255, 255, 255), "fg": (30, 30, 30), "ok": (30, 120, 60), "warn": (150, 90, 0), "crit": (190, 30, 30), "off": (90, 90, 90)},
          "야간": {"bg": (18, 22, 28), "fg": (225, 230, 235), "ok": (90, 210, 130), "warn": (245, 190, 70), "crit": (255, 110, 110), "off": (160, 165, 170)}}


def lum(c):
    # [TODO : 학생 실습 6] 오프라인/온라인 전환 상태 및 재고 변동 내역 실시간 대시보드 UI 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def contrast(a, b):
    la, lb = sorted([lum(a), lum(b)], reverse=True)
    return (la + 0.05) / (lb + 0.05)


class Dashboard:
    """화면 모델: 매 tick(1시간)마다 새 트랜잭션을 반영하고 4개 패널의 표시 값을 갱신한다. 렌더링은 모델과 분리해 응답 시간을 따로 잰다."""
    def __init__(self, txns, outages, base=500):
        self.tx = sorted(txns, key=lambda x: x["t"]); self.p = 0; self.outages = outages
        self.central = {i: base for i in ITEM_IDS}; self.local = {n: {i: 0 for i in ITEM_IDS} for n in NODES}      # 서버 반영 재고 / 노드별 미동기 변동
        self.pending = {n: 0 for n in NODES}; self.last_sync = {n: 0.0 for n in NODES}; self.feed = []; self.theme = "주간"; self.node_filter = None; self.only_alert = False; self.acked = set(); self.out24 = {i: [] for i in ITEM_IDS}

    def online(self, n, t):
        return not any(a <= t < b for a, b in self.outages.get(n, []))

    def step(self, t):
        while self.p < len(self.tx) and self.tx[self.p]["t"] < t:
            x = self.tx[self.p]; self.p += 1; d = x["qty"] if x["op"] in ("in", "adjust") else -x["qty"]
            if self.online(x["node"], x["t"]):
                self.central[x["item"]] += d; self.last_sync[x["node"]] = x["t"]
            else:
                self.local[x["node"]][x["item"]] += d; self.pending[x["node"]] += 1
            if x["op"] == "out":
                self.out24[x["item"]].append((x["t"], x["qty"]))
            self.feed.append((x["t"], x["node"], x["item"], x["op"], x["qty"], "반영" if self.online(x["node"], x["t"]) else "대기"))
        for n in NODES:                                                  # 복구된 노드는 대기분을 한꺼번에 반영
            if self.online(n, t) and self.pending[n]:
                for i, v in self.local[n].items():
                    self.central[i] += v; self.local[n][i] = 0
                self.pending[n] = 0; self.last_sync[n] = t
        return self.view(t)

    def view(self, t):
        status = [{"node": n, "state": "온라인" if self.online(n, t) else "오프라인", "pending": self.pending[n], "lag_h": round(t - self.last_sync[n], 1)} for n in NODES]
        gauges = []
        for it in ITEMS:
            st = self.central[it["item"]] + sum(self.local[n][it["item"]] for n in NODES)
            o24 = sum(q for (tt, q) in self.out24[it["item"]] if tt >= t - 24)
            cover = st / max(o24, 1) if t >= 24 else 99.0                      # 최근 24시간 불출량 기준 소진 예상 일수
            lvl = "긴급" if (st < it["min_level"] or cover < 0.5 * it["lead_days"]) else "주의" if cover < it["lead_days"] else "정상"
            gauges.append({"item": it["item"], "stock": st, "min": it["min_level"], "lvl": lvl, "cover": cover, "lead": it["lead_days"], "acked": it["item"] in self.acked})
        if self.only_alert:
            gauges = [g for g in gauges if g["lvl"] != "정상"]
        feed = [f for f in self.feed[-200:] if not self.node_filter or f[1] == self.node_filter][-6:]
        alerts = [g for g in gauges if g["lvl"] == "긴급" and not g["acked"]] + [dict(node=s["node"], lvl="장기 오프라인") for s in status if s["lag_h"] >= 48]
        return {"t": t, "status": status, "gauges": gauges, "feed": feed, "alerts": alerts, "theme": self.theme}

    # ---- 사용자 조작 핸들러 ----
    def set_theme(self, name):
        self.theme = name; return self.view(self.tx[self.p - 1]["t"] if self.p else 0)

    def set_node(self, n):
        self.node_filter = n; return self.view(self.tx[self.p - 1]["t"])

    def toggle_alert(self):
        self.only_alert = not self.only_alert; return self.view(self.tx[self.p - 1]["t"])

    def ack(self, item):
        self.acked.add(item); return self.view(self.tx[self.p - 1]["t"])


rng = np.random.default_rng(21)
TX = make_txns(7, 7, per_node_day=60)
OUT = {"보급소B": [(24.0, 96.0)], "보급소D": [(60.0, 120.0), (140.0, 150.0)], "보급소E": [(10.0, 168.5)]}        # 오프라인 구간(시간)
db = Dashboard(TX, OUT)
lat, snaps, off_n = [], {}, []
for h in range(1, 169):
    t0 = time.perf_counter(); v = db.step(float(h)); lat.append((time.perf_counter() - t0) * 1000)
    off_n.append(sum(s["state"] == "오프라인" for s in v["status"]))
    if h in (12, 72, 168):
        snaps[h] = v
lat = np.array(lat)
print(f"[모의 스트림] {len(TX)}건 트랜잭션, 168회 갱신(1시간 단위), 오프라인 구간 {sum(len(x) for x in OUT.values())}개")
print(f"화면 모델 갱신 지연: 평균 {lat.mean():.2f}ms, 95% {np.percentile(lat, 95):.2f}ms, 최대 {lat.max():.2f}ms (목표 1초 이내: {'달성' if lat.max() < 1000 else '미달'})")
for h, v in snaps.items():
    print(f"  {h:3d}h 시점: 오프라인 {sum(s['state'] == '오프라인' for s in v['status'])}곳, 미동기 {sum(s['pending'] for s in v['status'])}건, 긴급/주의 품목 {sum(g['lvl'] == '긴급' for g in v['gauges'])}/{sum(g['lvl'] == '주의' for g in v['gauges'])}종, 경보 {len(v['alerts'])}건")
last = snaps[168]
print("\n[보급소 상태판 168h]  " + " | ".join(f"{s['node'][-1]} {s['state']} 대기{s['pending']} 지연{s['lag_h']}h" for s in last["status"]))
print("[재고 경보 품목] " + ", ".join(f"{g['item']}:{g['stock']}/소진{g['cover']:.1f}일/납기{g['lead']}일({g['lvl']})" for g in [x for x in last["gauges"] if x["lvl"] != "정상"][:4]) + " ...")
print("[변동 내역] " + "; ".join(f"{f[1][-1]}-{f[2]} {f[3]} {f[4]} {f[5]}" for f in last["feed"][-3:]))

# ---- 사용자 조작 시험: 100회씩 응답 시간 ----
def timeit(fn, k=100):
    r = []
    for _ in range(k):
        t0 = time.perf_counter(); fn(); r.append((time.perf_counter() - t0) * 1000)
    return np.array(r)
ops = {"야간 모드 전환": lambda: (db.set_theme("야간"), db.set_theme("주간")), "보급소 필터": lambda: (db.set_node("보급소C"), db.set_node(None)),
       "경보만 보기": lambda: (db.toggle_alert(), db.toggle_alert()), "경보 확인(ack)": lambda: (db.ack("I01"), db.acked.clear())}
print("\n[UI 조작 응답 시간 ms] 조작 / 평균 / 95% / 최대")
worst = 0
for k, fn in ops.items():
    r = timeit(fn); worst = max(worst, r.max()); print(f"  {k:12s} {r.mean():6.3f} {np.percentile(r, 95):6.3f} {r.max():6.3f}")
print(f"-> 최대 {worst:.2f}ms (목표 100ms 이내: {'달성' if worst < 100 else '미달'})")

# ---- 야간 모드 가독성(WCAG 대비 4.5 이상)과 긴급 경보 강조 ----
print("\n[테마 대비비(글자-배경)]")
ok = True
for nm, th in THEMES.items():
    cs = {k: contrast(th[k], th["bg"]) for k in ("fg", "ok", "warn", "crit", "off")}; ok &= all(c >= 4.5 for c in cs.values())
    print(f"  {nm}: " + ", ".join(f"{k} {c:.1f}" for k, c in cs.items()))
print(f"모든 색상 대비 4.5 이상: {'달성' if ok else '미달'}")

# ---- 렌더링(4개 패널) ----
th = THEMES["야간"]; col = lambda k: tuple(c / 255 for c in th[k])
t0 = time.perf_counter()
fig, ax = plt.subplots(2, 2, figsize=(10, 6), facecolor=col("bg"))
for a in ax.ravel():
    a.set_facecolor(col("bg")); a.tick_params(colors=col("fg")); [s.set_color(col("off")) for s in a.spines.values()]
a = ax[0, 0]; a.set_title("Node status (pending tx)", color=col("fg"))
a.bar([s["node"][-1] for s in last["status"]], [s["pending"] for s in last["status"]], color=[col("ok") if s["state"] == "온라인" else col("crit") for s in last["status"]])
a = ax[0, 1]; a.set_title("Stock vs minimum level", color=col("fg")); ys = np.arange(len(last["gauges"]))
a.barh(ys, [g["stock"] for g in last["gauges"]], color=[col({"정상": "ok", "주의": "warn", "긴급": "crit"}[g["lvl"]]) for g in last["gauges"]])
a.scatter([g["min"] for g in last["gauges"]], ys, color=col("fg"), marker="|", s=80); a.set_yticks(ys); a.set_yticklabels([g["item"] for g in last["gauges"]], fontsize=6)
a = ax[1, 0]; a.set_title("Offline nodes over time", color=col("fg")); a.step(range(1, 169), off_n, color=col("warn")); a.set_xlabel("hour", color=col("fg"))
a = ax[1, 1]; a.set_title("Alert count by stage", color=col("fg"))
a.bar(["critical", "caution", "normal"], [sum(g["lvl"] == k for g in last["gauges"]) for k in ("긴급", "주의", "정상")], color=[col("crit"), col("warn"), col("ok")])
fig.tight_layout(); fig.canvas.draw(); rt = (time.perf_counter() - t0) * 1000
print(f"\n렌더링 4개 패널 1회 그리기 {rt:.0f}ms (갱신 1초 이내: {'달성' if rt + lat.max() < 1000 else '미달'}, 응답성 100ms는 화면 모델 조작 기준)")
print("※ 대시보드는 판단을 대신하지 않는다. 경보 확인(ack)과 재고 조치는 군수관이 한다. 화면의 재고는 동기화된 값과 노드별 미동기 변동의 합이다.")
plt.show()


<details>
<summary><b>💡 [참고] 미션 06 예상 실행 결과 (Output)</b></summary>

```text
[모의 스트림] 2100건 트랜잭션, 168회 갱신(1시간 단위), 오프라인 구간 4개
화면 모델 갱신 지연: 평균 0.14ms, 95% 0.20ms, 최대 1.36ms (목표 1초 이내: 달성)
   12h 시점: 오프라인 1곳, 미동기 6건, 긴급/주의 품목 0/0종, 경보 0건
   72h 시점: 오프라인 3곳, 미동기 314건, 긴급/주의 품목 7/3종, 경보 9건
  168h 시점: 오프라인 1곳, 미동기 400건, 긴급/주의 품목 6/3종, 경보 7건

[보급소 상태판 168h]  A 온라인 대기0 지연1.6h | B 온라인 대기0 지연0.1h | C 온라인 대기0 지연0.2h | D 온라인 대기0 지연0.2h | E 오프라인 대기400 지연160.4h
[재고 경보 품목] I02:679/소진2.0일/납기8일(긴급), I03:524/소진3.8일/납기13일(긴급), I04:997/소진2.5일/납기6일(긴급), I05:1503/소진4.9일/납기11일(긴급) ...
[변동 내역] C-I05 out 15 반영; C-I03 in 42 반영; B-I02 in 87 반영

[UI 조작 응답 시간 ms] 조작 / 평균 / 95% / 최대
  야간 모드 전환      0.423  0.498  1.075
  보급소 필터        0.467  0.503  0.627
  경보만 보기        0.462  0.497  0.640
  경보 확인(ack)    0.121  0.149  0.218
-> 최대 1.08ms (목표 100ms 이내: 달성)

[테마 대비비(글자-배경)]
  주간: fg 16.7, ok 5.5, warn 5.6, crit 6.2, off 6.9
  야간: fg 14.4, ok 9.5, warn 10.7, crit 6.7, off 7.3
모든 색상 대비 4.5 이상: 달성

렌더링 4개 패널 1회 그리기 349ms (갱신 1초 이내: 달성, 응답성 100ms는 화면 모델 조작 기준)
※ 대시보드는 판단을 대신하지 않는다. 경보 확인(ack)과 재고 조치는 군수관이 한다. 화면의 재고는 동기화된 값과 노드별 미동기 변동의 합이다.
```
</details>
